# What Is Associated with NBA Player Improvement?

This project compares NBA players year to year and asks which factors (change in scoring, assists, efficiency, usage, and age) are associated with improvement. It then tests whether the answer depends on how "improvement" is defined.

**Note:** Run every cell from top to bottom. The Colab upload cell asks for `all_seasons.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()

First, load the dataset and look at its structure to confirm which variables are available.

In [ ]:
import pandas as pd

df = pd.read_csv("all_seasons.csv")
df.head()

Record the raw dataset coverage before any rows are removed. These numbers go in the README's Data section. (Do this *before* dropping rows: removing players' last seasons later also removes the final season in the file.)

In [ ]:
print("Rows:", len(df))
print("Unique players:", df["player_name"].nunique())
print("First season:", df["season"].min())
print("Last season:", df["season"].max())

Remove the unnecessary index column left over from the CSV export.

In [ ]:
df = df.drop(columns=["Unnamed: 0"])

Sort by player and season so each player's seasons are in chronological order. This is required before aligning seasons.

In [ ]:
df = df.sort_values(by=["player_name", "season"])

Create "next season" columns for each player so a player's current season can be compared with the following one.

`shift(-1)` moves each stat up one row within each player's group. Note that this grabs the player's next *recorded* season, which is not always the next calendar season (checked in an optional cell near the end).

In [ ]:
g = df.groupby("player_name")
df["pts_next"] = g["pts"].shift(-1)
df["ast_next"] = g["ast"].shift(-1)
df["ts_pct_next"] = g["ts_pct"].shift(-1)
df["usg_pct_next"] = g["usg_pct"].shift(-1)
df["gp_next"] = g["gp"].shift(-1)
df["season_next"] = g["season"].shift(-1)

Quick check that alignment worked: each row should show current and next-season stats.

In [ ]:
df[["player_name", "season", "pts", "pts_next"]].head(10)

Remove rows with no next season (usually a player's final recorded season).

In [ ]:
df = df.dropna(subset=["pts_next"])
print("Player-season pairs:", len(df))

## Run 1: hand-weighted improvement score

The score combines year-over-year changes in scoring, playmaking, efficiency (TS%), and usage.

**The weights were chosen by hand, with extra weight on efficiency.** Because TS% change is part of the score and was given a large weight, a strong correlation between TS% change and the score is partly built in. Run 2 tests this.

In [ ]:
df["improvement"] = (
    (df["pts_next"] - df["pts"]) +
    0.5 * (df["ast_next"] - df["ast"]) +
    10 * (df["ts_pct_next"] - df["ts_pct"]) +
    0.3 * (df["usg_pct_next"] - df["usg_pct"])
)

Display a few rows to confirm the score is calculated correctly.

In [ ]:
df[["player_name", "season", "improvement"]].head()

How does age relate to improvement? Do younger players tend to improve more than older players?

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df["age"], df["improvement"])
plt.xlabel("Age")
plt.ylabel("Improvement")
plt.title("Age vs Improvement")
plt.show()

Calculate how a player's usage changes between seasons (usage is how involved a player is in the offense), and check whether usage change is associated with improvement.

In [ ]:
df["usg_change"] = df["usg_pct_next"] - df["usg_pct"]

plt.scatter(df["usg_change"], df["improvement"])
plt.xlabel("Usage Change")
plt.ylabel("Improvement")
plt.title("Usage Change vs Improvement")
plt.show()

Group players by age and plot the average improvement, to look for trends in when players tend to peak or decline.

In [ ]:
age_group = df.groupby("age")["improvement"].mean()

age_group.plot()
plt.xlabel("Age")
plt.ylabel("Average Improvement")
plt.title("Improvement by Age")
plt.show()

Check how many player-seasons exist at each age. Very small samples (ages 18 and 39+) can create misleading spikes in the graph above.

In [ ]:
df["age"].value_counts().sort_index()

Correlation between age and improvement. A negative value means players tend to improve less as they get older.

In [ ]:
df[["age", "improvement"]].corr()

Correlation between usage change and improvement.

In [ ]:
df[["usg_change", "improvement"]].corr()

Calculate efficiency change (TS% difference between seasons) and its correlation with improvement.

In [ ]:
df["ts_change"] = df["ts_pct_next"] - df["ts_pct"]

df[["ts_change", "improvement"]].corr()

Combine the variables into one correlation table.

In [ ]:
df[["age", "usg_change", "ts_change", "improvement"]].corr()

Fit a linear regression predicting improvement from age, usage change, and TS% change.

**The raw coefficients below cannot be compared with each other**, because the variables are in different units (TS% and usage change are small decimals, age is in years). The next cell standardizes everything so the coefficients are comparable.

In [ ]:
from sklearn.linear_model import LinearRegression

X = df[["age", "usg_change", "ts_change"]]
y = df["improvement"]

model = LinearRegression()
model.fit(X, y)

print("Raw coefficients (age, usg_change, ts_change):", model.coef_)

In [ ]:
z = lambda s: (s - s.mean()) / s.std()

Xz = X.apply(z)
yz = z(y)

model_z = LinearRegression()
model_z.fit(Xz, yz)

print("Standardized coefficients (age, usg_change, ts_change):", model_z.coef_.round(3))
print("R-squared:", round(model_z.score(Xz, yz), 3))

## Run 2: robustness check (equal weights, minimum games)

Run 1's result depends on the hand-picked weights. To test that, build a second score that adds the four standardized changes with equal weight, and keep only player-seasons with enough games played in both seasons. The function is reused below to test several games cutoffs.

In [ ]:
def run2(min_gp):
    d = df[(df["gp"] >= min_gp) & (df["gp_next"] >= min_gp)].copy()
    d["pts_chg"] = d["pts_next"] - d["pts"]
    d["ast_chg"] = d["ast_next"] - d["ast"]
    d["improve_eq"] = z(d["pts_chg"]) + z(d["ast_chg"]) + z(d["ts_change"]) + z(d["usg_change"])
    cols = ["pts_chg", "ast_chg", "ts_change", "usg_change", "age", "improve_eq"]
    return d, d[cols].corr()["improve_eq"]

df2, corr20 = run2(20)
print("Player-season pairs (20+ games both seasons):", len(df2))
print("Unique players:", df2["player_name"].nunique())
print()
print(corr20.round(3))

Does the conclusion depend on the 20-game cutoff? Repeat with other cutoffs.

In [ ]:
rows = {}
for gp in [10, 20, 40]:
    d, c = run2(gp)
    rows[f"{gp}+ games (n={len(d)})"] = c.drop("improve_eq")

pd.DataFrame(rows).round(3)

Optional diagnostic: how many matched pairs skip a season (for example, a player who missed a year)? Those pairs compare seasons that are not consecutive.

In [ ]:
start = lambda s: s.str[:4].astype(int)
gap = start(df["season_next"]) - start(df["season"])

print(gap.value_counts().sort_index())
print("Share of pairs that are not consecutive seasons:", round((gap != 1).mean(), 4))

## Takeaways

- In Run 1, TS% change correlated most strongly with the improvement score, but the weights were chosen to favor efficiency, so this result is partly built in.
- In Run 2 (equal weights), change in points correlated most strongly with the score, and TS% change was the weakest of the four components. The ranking depends on how the score is weighted.
- Age is not part of the score, and its correlation with improvement was about -0.3 in both runs: older players tended to improve less. This is the one result that held across versions.
- All of these are correlations with a score made from the same statistics, so they do not show what causes improvement. A stronger test would use an outcome that is not part of the score, such as next-season minutes, win shares, or box plus/minus.